# Processing
Process data frames from interim parquet files created from original stat result files.
This notebook requires interim processing from  `mininet_vsomeip_dnssec/dataset.py`.
Then this notebook is used to look inside the data while processing it to finalize it.
The goal is to provide fully aggregated stats used for plotting and table generation in the evaluation report. 

The report looks at the following metrics:
- Service setup (min, mean, max) -- time to set up a connection between one publisher and its subscriber.
- Network initialization (min, mean, max) -- time to set up the network and establish all publisher and subscriber connections.
- Crypto Any (min, mean, max) -- time to perform any cryptographic operation for a message (sign and verify).
    - Create signature(min, mean, max) -- time to create any signature for a message.
    - Verify signature (min, mean, max) -- time to verify any signature for a message.
- Resolve Any DNS (min, mean, max) -- time to resolve any DNS record for a publisher or subscriber (SVCB and TLSA).
    - Resolve Pub SVCB (min, mean, max) -- time to resolve the service name to an IP address and port number for a publisher.
    - Resolve Pub TLSA (min, mean, max) -- time to resolve the TLSA record for a publisher.
    - Resolve Sub TLSA (min, mean, max) -- time to resolve the TLSA record for a subscriber.

In [143]:
import polars as pl
import os
from mininet_vsomeip_dnssec.config import INTERIM_DATA_DIR, SCENARIOS
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(-1)

polars.config.Config

In [144]:
# select an example config to look at
SCENARIO = "carnet"
SERIES = "H"
CONFIG = "p212_s448"
FILE = "all_runs.parquet"
interim_path = INTERIM_DATA_DIR / SCENARIO / SERIES / CONFIG / FILE
df = pl.read_parquet(interim_path)
df.select("offer_to_subscribe_dur_max", "run_num") #.select("offer_to_subscribe_dur_min", "offer_receive_to_subscribeack_dur_min", "offer_receive_to_verify_service_dur_min", "subscription_dur_min", "run_num")

offer_to_subscribe_dur_max,run_num
i64,i32
123212069,1
60124298,2
38255755,3
42379141,4
51124256,5
51595227,6
350758719,7
101969151,8
65514916,9


In [145]:
# look at a specific run to see all the data before aggregation
run_num = 18
run_file = f"run-{run_num}.parquet"
run_interim_path = INTERIM_DATA_DIR / SCENARIO / SERIES / CONFIG / run_file
df_run = pl.read_parquet(run_interim_path)
df_run #.select("offer_to_subscribe_dur_min", "offer_receive_to_subscribeack_dur_min", "offer_receive_to_verify_service_dur_min", "subscription_dur_min", "valid_offer_to_subscription_dur_min")

subscriber_app_init_dur_min,subscriber_app_init_dur_mean,subscriber_app_init_dur_stddev,subscriber_app_init_dur_max,find_offer_dur_min,find_offer_dur_mean,find_offer_dur_stddev,find_offer_dur_max,svcb_service_dur_min,svcb_service_dur_mean,svcb_service_dur_stddev,svcb_service_dur_max,validate_offer_dur_min,validate_offer_dur_mean,validate_offer_dur_stddev,validate_offer_dur_max,client_sign_dur_min,client_sign_dur_mean,client_sign_dur_stddev,client_sign_dur_max,offer_to_subscribe_dur_min,offer_to_subscribe_dur_mean,offer_to_subscribe_dur_stddev,offer_to_subscribe_dur_max,subscribe_transmission_dur_min,subscribe_transmission_dur_mean,subscribe_transmission_dur_stddev,subscribe_transmission_dur_max,tlsa_client_dur_min,tlsa_client_dur_mean,tlsa_client_dur_stddev,tlsa_client_dur_max,verify_client_dur_min,verify_client_dur_mean,verify_client_dur_stddev,verify_client_dur_max,service_sign_dur_min,service_sign_dur_mean,service_sign_dur_stddev,service_sign_dur_max,subscribeack_transmission_dur_min,subscribeack_transmission_dur_mean,subscribeack_transmission_dur_stddev,subscribeack_transmission_dur_max,subscribe_to_subscribeack_client_dur_min,subscribe_to_subscribeack_client_dur_mean,subscribe_to_subscribeack_client_dur_stddev,subscribe_to_subscribeack_client_dur_max,subscribe_to_subscribeack_service_dur_min,subscribe_to_subscribeack_service_dur_mean,subscribe_to_subscribeack_service_dur_stddev,subscribe_to_subscribeack_service_dur_max,tlsa_service_dur_min,tlsa_service_dur_mean,tlsa_service_dur_stddev,tlsa_service_dur_max,verify_service_dur_min,verify_service_dur_mean,verify_service_dur_stddev,verify_service_dur_max,offer_receive_to_subscribeack_dur_min,offer_receive_to_subscribeack_dur_mean,offer_receive_to_subscribeack_dur_stddev,offer_receive_to_subscribeack_dur_max,validate_offer_to_subscribeack_dur_min,validate_offer_to_subscribeack_dur_mean,validate_offer_to_subscribeack_dur_stddev,validate_offer_to_subscribeack_dur_max,offer_receive_to_verify_service_dur_min,offer_receive_to_verify_service_dur_mean,offer_receive_to_verify_service_dur_stddev,offer_receive_to_verify_service_dur_max,validate_offer_to_verify_service_dur_min,validate_offer_to_verify_service_dur_mean,validate_offer_to_verify_service_dur_stddev,validate_offer_to_verify_service_dur_max,subscription_dur_min,subscription_dur_mean,subscription_dur_stddev,subscription_dur_max,valid_offer_to_subscription_dur_min,valid_offer_to_subscription_dur_mean,valid_offer_to_subscription_dur_stddev,valid_offer_to_subscription_dur_max,dns_resolution_dur_sum_min,dns_resolution_dur_sum_mean,dns_resolution_dur_sum_max,sign_dur_sum_min,sign_dur_sum_mean,sign_dur_sum_max,verify_dur_sum_min,verify_dur_sum_mean,verify_dur_sum_max,crypto_dur_sum_min,crypto_dur_sum_mean,crypto_dur_sum_max,total_offer_receive_to_subscribeack_dur,total_offer_receive_to_verify_service_dur,total_validate_offer_to_subscribeack_dur,total_validate_offer_to_verify_service_dur,total_dur
i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,i64,i64,f64,i64,i64,f64,i64,i64,f64,i64,i64,i64,i64,i64,i64
1524964,1.0596e7,3.4994e6,13425695,259405,4.4183e8,2.4332e8,1053787976,96174,3.4774e6,6.0642e6,51117066,64,145.174107,80.5216,622,521430,789637.267857,1.8572e6,38435290,577886,1.7324e7,1.2525e7,117873819,61638,3.1094e7,2.8033e7,95341541,127385,7.1218e6,8.0299e6,41066122,87319,119432.091518,44183.585522,304748,525110,725090.642857,517283.60342,4495615,48680,1.5058e7,2.1385e7,131464786,1494325,5.8781e7,3.9014e7,318360181,1344093,1.2630e7,1.6906e7,265047809,62303,3.1670e6,4.4873e6,23177349,80355,112747.316964,54265.926567,808027,3855159,7.6225e7,4.4144e7,328103459,3851129,7.6061e7,4.4218e7,328094649,3971440,7.6344e7,4.4140e7,328222528,3967410,7.6181

In [146]:
# list all column names, drop min mean stddev and max from the names and only show the metric name without the suffixes once
unique_metrics = [col.rstrip('_min').rstrip('_mean').rstrip('_stddev').rstrip('_max') for col in df.columns]
unique_metrics = list(sorted(set(unique_metrics)))
unique_metrics

['client_sign_dur',
 'crypto_dur_su',
 'dns_resolution_dur_su',
 'find_offer_dur',
 'offer_receive_to_subscribeack_dur',
 'offer_receive_to_verify_service_dur',
 'offer_to_subscribe_dur',
 'run_nu',
 'service_sign_dur',
 'sign_dur_su',
 'subscribe_to_subscribeack_client_dur',
 'subscribe_to_subscribeack_service_dur',
 'subscribe_transmission_dur',
 'subscribeack_transmission_dur',
 'subscriber_app_init_dur',
 'subscription_dur',
 'svcb_service_dur',
 'tlsa_client_dur',
 'tlsa_service_dur',
 'total_dur',
 'total_offer_receive_to_subscribeack_dur',
 'total_offer_receive_to_verify_service_dur',
 'total_validate_offer_to_subscribeack_dur',
 'total_validate_offer_to_verify_service_dur',
 'valid_offer_to_subscription_dur',
 'validate_offer_dur',
 'validate_offer_to_subscribeack_dur',
 'validate_offer_to_verify_service_dur',
 'verify_client_dur',
 'verify_dur_su',
 'verify_service_dur']

In [147]:
processed_columns = {
    "crypto_sum": "crypto_dur_sum",
    "create_signatures_sum": "sign_dur_sum",
    "verify_signatures_sum": "verify_dur_sum",
    "resolve_dns_sum": "dns_resolution_dur_sum",
    "resolve_pub_svcb": "svcb_service_dur",
    "resolve_pub_tlsa": "tlsa_service_dur",
    "resolve_sub_tlsa": "tlsa_client_dur", 
}
non_aggregates = {
    "network_initialization": "total_dur",
    "service_setup": "subscription_dur_max"
}
processed_results_cols = list(processed_columns.keys()) + list(non_aggregates.keys())
# add min max mean suffix
processed_results_cols = [col + suffix for col in processed_results_cols for suffix in ["_min", "_mean", "_stddev", "_max"]]

In [148]:
process_exprs = [
    expr
    for col_name, stat_name in processed_columns.items()
    for expr in [
        pl.min(f"{stat_name}_min").alias(f"{col_name}_min"),
        pl.mean(f"{stat_name}_mean").alias(f"{col_name}_mean"),
        pl.max(f"{stat_name}_max").alias(f"{col_name}_max"),
    ]
]
# add non aggregate
process_exprs.extend(
    [
        expr
        for col_name, stat_name in non_aggregates.items()
        for expr in [
            pl.min(f"{stat_name}").alias(f"{col_name}_min"),
            pl.mean(f"{stat_name}").alias(f"{col_name}_mean"),
            pl.max(f"{stat_name}").alias(f"{col_name}_max"),
        ]
    ]
)

In [151]:
processed_df = df.select(process_exprs)
processed_df.head()
print(processed_df.columns)

['crypto_sum_min', 'crypto_sum_mean', 'crypto_sum_max', 'create_signatures_sum_min', 'create_signatures_sum_mean', 'create_signatures_sum_max', 'verify_signatures_sum_min', 'verify_signatures_sum_mean', 'verify_signatures_sum_max', 'resolve_dns_sum_min', 'resolve_dns_sum_mean', 'resolve_dns_sum_max', 'resolve_pub_svcb_min', 'resolve_pub_svcb_mean', 'resolve_pub_svcb_max', 'resolve_pub_tlsa_min', 'resolve_pub_tlsa_mean', 'resolve_pub_tlsa_max', 'resolve_sub_tlsa_min', 'resolve_sub_tlsa_mean', 'resolve_sub_tlsa_max', 'network_initialization_min', 'network_initialization_mean', 'network_initialization_max', 'service_setup_min', 'service_setup_mean', 'service_setup_max']


In [150]:
# draw a matrix table with columns Metrics, min[ms], mean[ms], max[ms] and rows for each metric base name in [processed_columns + non_aggregates]
# the values in the table should be the corresponding values from processed_df with appropriate formatting to show the metric name and the min mean max values in ms with 2 decimal places. You can use the following code to format the values: f"{value:.2f} ms"
metric_names = list(processed_columns.keys()) + list(non_aggregates.keys())
table_data = [["Metric", "min[ms]", "mean[ms]", "max[ms]"]]

for metric in metric_names:
    row = [metric]
    for suffix in ["_min", "_mean", "_max"]:
        value = processed_df.select(f"{metric}{suffix}").item()
        if value is None:
            row.append("n/a")
        else:
            row.append(f"{float(value) / 1_000_000:.2f} ms")
    table_data.append(row)

for row in table_data:
    print("{:<30} {:<10} {:<10} {:<10}".format(*row))



Metric                         min[ms]    mean[ms]   max[ms]   
crypto_sum                     1.23 ms    1.76 ms    50.01 ms  
create_signatures_sum          1.05 ms    1.51 ms    49.68 ms  
verify_signatures_sum          0.17 ms    0.24 ms    12.53 ms  
resolve_dns_sum                0.31 ms    18.04 ms   168.42 ms 
resolve_pub_svcb               0.05 ms    4.68 ms    109.84 ms 
resolve_pub_tlsa               0.04 ms    4.05 ms    100.65 ms 
resolve_sub_tlsa               0.08 ms    9.31 ms    71.29 ms  
network_initialization         1165.36 ms 1234.38 ms 1298.39 ms
service_setup                  123.51 ms  226.25 ms  455.61 ms 
